In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
import matplotlib.pyplot as plt
import numpy as np
from matplotlib.path import Path
import pandas as pd
from torch.optim.lr_scheduler import CosineAnnealingLR
import copy
import time 
import torch.nn.functional as F

# Set random seed for reproducibility
torch.manual_seed(0)
np.random.seed(0)

# Check if GPU is available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {device}")

# Set default tensor type to float32
torch.set_default_tensor_type(torch.FloatTensor)

# ANN

In [ ]:
class Net(nn.Module):
    def __init__(self):
        super(Net, self).__init__()
        self.hidden1 = nn.Linear(2, 128)
        self.hidden2 = nn.Linear(128, 128)
        self.hidden3 = nn.Linear(128, 128)
        self.hidden4 = nn.Linear(128, 128)  
        self.hidden5 = nn.Linear(128, 128)
        self.output = nn.Linear(128, 2)
    def forward(self, x):
        x = torch.tanh(self.hidden1(x))
        x = torch.tanh(self.hidden2(x))
        x = torch.tanh(self.hidden3(x))
        x = torch.tanh(self.hidden4(x))
        x = torch.tanh(self.hidden5(x))
        return self.output(x)  # return [u, v]
    # Define the problem domain using the given vertices

# Vertice

In [ ]:
from matplotlib.path import Path
import numpy as np

R = 1.0  # Radius

verts = [
    (R, 0),      # Start (1,0)
    (4, 0),      # ขวาล่าง
    (4, 4),      # ขวาบน
    (0, 4),      # ซ้ายบน
    (0, R),      # ลงมาถึง (0,1)
    # Bezier approximation ของ arc 90°: control point ≈ (0, 0.5523), (0.5523, 0)
    (0, R * (1 - 0.5523)),   # (0, 0.4477) — control point 1
    (R * (1 - 0.5523), 0),   # (0.4477, 0) — control point 2
    (R, 0),                  # ปลาย arc กลับมา (1,0)
    (R, 0),                  # CLOSEPOLY
]

codes = [
    Path.MOVETO,
    Path.LINETO,
    Path.LINETO,
    Path.LINETO,
    Path.LINETO,
    Path.CURVE4,   # arc ด้วย cubic bezier (ต้องการ 3 จุด)
    Path.CURVE4,
    Path.CURVE4,
    Path.CLOSEPOLY,
]

path = Path(verts, codes)

def in_domain(x, y):
    points = np.column_stack((x.cpu().numpy(), y.cpu().numpy()))
    return torch.tensor(path.contains_points(points), dtype=torch.bool, device=device)

# BC

In [ ]:
def BC_bottom(x, y):
    tol = 1e-6
    return ((torch.abs(y - 0) < tol) & (x >= 1.0) & (x <= 4.0)).squeeze()  # ← เริ่มที่ x=1 (ตัด arc ออก)

def BC_left(x, y):
    tol = 1e-6
    return ((torch.abs(x - 0) < tol) & (y >= 1.0) & (y <= 4.0)).squeeze()  # ← เริ่มที่ y=1

def BC_top(x, y):
    tol = 1e-6
    return ((torch.abs(y - 4) < tol) & (x >= 0) & (x <= 4.0)).squeeze()   # ← x ถึง 4.0

def BC_right(x, y):
    tol = 1e-6
    return ((torch.abs(x - 4.0) < tol) & (y >= 0) & (y <= 4.0)).squeeze()  # ← x=4.0

def BC_curved(x, y):
    """ขอบโค้ง: จุดที่อยู่บน arc x²+y²=1, ช่วง 0°–90°"""
    tol = 0.005   # tolerance สำหรับเส้นโค้ง
    r = torch.sqrt(x**2 + y**2)
    return ((torch.abs(r - 1.0) < tol) & (x >= 0) & (y >= 0) & (x <= 1.0) & (y <= 1.0)).squeeze()

# BC Loss

In [ ]:
# ============================================================
# Material & load constants  (single source of truth)
# ============================================================
E   = 20.0  # Young's modulus [MPa]
nu  = 0.3    # Poisson's ratio
R   = 1.0    # Tunnel radius [m]

# Plane-Strain constitutive pre-factors
C11 = E * (1 - nu) / ((1 + nu) * (1 - 2 * nu))   # σ_xx / ε_xx  (plane-strain)
C12 = E * nu       / ((1 + nu) * (1 - 2 * nu))   # σ_xx / ε_yy  (plane-strain)
C33 = E            / (2 * (1 + nu))               # σ_xy / γ_xy

# Distributed load on top edge  x ∈ [LOAD_X0, LOAD_X1]
LOAD_X0  = 0   # [m]  ← must match generate_point_distribution()
LOAD_X1  = 4.0   # [m]
LOAD_VAL = -0.120  # [MPa]  (compression → negative)


def _stress_from_grad(xy_pts, net):
    """Compute plane-strain stresses at xy_pts (requires_grad must be set)."""
    out  = net(xy_pts)
    u, v = out[:, 0:1], out[:, 1:2]
    grads_u = torch.autograd.grad(u.sum(), xy_pts, create_graph=True)[0]
    grads_v = torch.autograd.grad(v.sum(), xy_pts, create_graph=True)[0]
    u_x, u_y = grads_u[:, 0], grads_u[:, 1]
    v_x, v_y = grads_v[:, 0], grads_v[:, 1]
    s_xx = C11 * u_x + C12 * v_y
    s_yy = C12 * u_x + C11 * v_y
    s_xy = C33 * (u_y + v_x)
    return s_xx, s_yy, s_xy


def BC(xy, net):
    """
    Boundary-condition loss for the quarter-domain with a circular cutout.

    Boundaries
    ----------
    Bottom  (y=0, x∈[R,4])  : u_x = 0, u_y = 0  (roller/fixed)
    Left    (x=0, y∈[R,4])  : u = 0, σ_xy = 0   (symmetry)
    Right   (x=4, y∈[0,4])  : u = 0, σ_xy = 0   (symmetry)         
    Top     (y=4, x∈[0,4])  : σ_xy = 0          (free shear everywhere)
                               σ_yy = LOAD_VAL    on [LOAD_X0, LOAD_X1]
                               σ_yy = 0           elsewhere
    Curved  (x²+y²=R²)      : σ_n  = 0, σ_t = 0  (traction-free tunnel wall)
    """
    x = xy[:, 0].unsqueeze(1)
    y = xy[:, 1].unsqueeze(1)

    out = net(xy)
    u   = out[:, 0:1]
    v   = out[:, 1:2]

    # ── Boundary masks ──────────────────────────────────────────────────────
    bc_b = BC_bottom(x, y)   # bottom
    bc_l = BC_left  (x, y)   # left
    bc_r = BC_right (x, y)   # right
    bc_t = BC_top   (x, y)   # top
    bc_c = BC_curved(x, y)   # curved tunnel wall

    loss = torch.tensor(0.0, device=xy.device)

    # ── Bottom: v = 0 ────────────────────────────────────────────────
    if bc_b.any():
        loss += F.mse_loss(v[bc_b], torch.zeros_like(v[bc_b]))   # uy = 0 ✅
        xy_b = xy[bc_b].detach().requires_grad_(True)
        _, _, s_xy_b = _stress_from_grad(xy_b, net)
        loss += F.mse_loss(s_xy_b, torch.zeros_like(s_xy_b))

    # ── Left: u = 0, σ_xy = 0 (symmetry) ────────────────────────────────────
    if bc_l.any():
        # Normal displacement is zero
        loss += F.mse_loss(u[bc_l], torch.zeros_like(u[bc_l]))
        
        # Shear stress is zero
        xy_l = xy[bc_l].detach().requires_grad_(True)
        _, _, s_xy_l = _stress_from_grad(xy_l, net)
        loss += F.mse_loss(s_xy_l, torch.zeros_like(s_xy_l))

# ── Right: σ_xy = 0  +  u = 0 ────────────────────────────
    if bc_r.any():
    # u = 0 (horizontal displacement fixed)
        loss += F.mse_loss(u[bc_r], torch.zeros_like(u[bc_r]))

    # shear stress = 0
        xy_r = xy[bc_r].detach().requires_grad_(True)
        _, _, s_xy_r = _stress_from_grad(xy_r, net)
        loss += F.mse_loss(s_xy_r, torch.zeros_like(s_xy_r))

    # ── Top: mixed Neumann ────────────────────────────────────────────────────
    if bc_t.any():
        xy_t  = xy[bc_t].detach().requires_grad_(True)
        x_t   = xy_t[:, 0]
        s_xx_t, s_yy_t, s_xy_t = _stress_from_grad(xy_t, net)

        # σ_xy = 0  everywhere on top
        loss += F.mse_loss(s_xy_t, torch.zeros_like(s_xy_t))

        # σ_yy = LOAD_VAL under the footing, 0 elsewhere
        load_mask    = ((x_t >= LOAD_X0) & (x_t <= LOAD_X1))
        no_load_mask = ~load_mask

        if no_load_mask.any():
            loss += F.mse_loss(s_yy_t[no_load_mask],
                               torch.zeros_like(s_yy_t[no_load_mask]))
        if load_mask.any():
            loss += F.mse_loss(s_yy_t[load_mask],
                               LOAD_VAL * torch.ones_like(s_yy_t[load_mask]))

    # ── Curved wall: traction-free  σ_n = 0, σ_t = 0 ─────────────────────────
    # Outward normal on x²+y²=R² pointing inward toward origin: n = -(x,y)/R
    # For the tunnel (hole), outward normal = -(x,y)/R
    if bc_c.any():
        xy_c  = xy[bc_c].detach().requires_grad_(True)
        x_c   = xy_c[:, 0]
        y_c   = xy_c[:, 1]
        r_c   = torch.sqrt(x_c**2 + y_c**2).clamp(min=1e-8)
        
        # Depending on convention, for a hole the normal points towards the origin:
        nx_c = x_c / r_c   # outward จากดิน = ชี้เข้าอุโมงค์
        ny_c = y_c / r_c

        s_xx_c, s_yy_c, s_xy_c = _stress_from_grad(xy_c, net)

        # Traction vector t = σ · n
        t_x = s_xx_c * nx_c + s_xy_c * ny_c   # normal traction
        t_y = s_xy_c * nx_c + s_yy_c * ny_c   # shear  traction

        loss += F.mse_loss(t_x, torch.zeros_like(t_x))
        loss += F.mse_loss(t_y, torch.zeros_like(t_y))

    return loss

# Datapoint sampling

In [ ]:
import torch
import numpy as np
import matplotlib.pyplot as plt

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

def generate_point_distribution(nx=100, ny=100, arc_density=20, arc_influence=0.8):
    x_min, x_max = 0.0, 4.0
    y_min, y_max = 0.0, 4.0
    R = 1.0

    # 1. สร้าง random points เริ่มต้น
    num_random_points = nx * ny * 5
    x_random = torch.rand(num_random_points, device=device) * (x_max - x_min) + x_min
    y_random = torch.rand(num_random_points, device=device) * (y_max - y_min) + y_min

    # 2. กรองจุดที่อยู่ในรู (x²+y² < R² บริเวณมุมล่างซ้าย)
    dist_from_origin = torch.sqrt(x_random**2 + y_random**2)
    in_corner_zone = (x_random < R) & (y_random < R)
    invalid_mask = in_corner_zone & (dist_from_origin < R)

    valid_mask = ~invalid_mask
    x_random = x_random[valid_mask]
    y_random = y_random[valid_mask]
    dist_from_origin = dist_from_origin[valid_mask]

    # 3. คำนวณระยะห่างจาก arc
    dist_to_arc = torch.abs(dist_from_origin - R)

    # 4. เฉพาะบริเวณใกล้ arc (แต่ไม่ใช่ ON arc)
    arc_weight_mask = (x_random < R * 1.5) & (y_random < R * 1.5)
    not_on_arc = dist_to_arc > 0.01  # ← กัน interior points ไม่ให้ติด boundary arc

    # 5. Probability
    base_prob = 0.4  # เพิ่มจาก 0.2 → จุดทั่ว domain สม่ำเสมอขึ้น
    prob_arc = arc_influence / (1 + (dist_to_arc * arc_density) ** 2)
    prob_arc = prob_arc * arc_weight_mask.float() * not_on_arc.float()  # ← กัน ON arc

    keep_prob = torch.clamp(base_prob + prob_arc, 0, 1)

    # 6. สุ่มเลือกจุด
    keep_mask = torch.rand(len(x_random), device=device) < keep_prob
    x_final = x_random[keep_mask]
    y_final = y_random[keep_mask]

    # --- Plot ---
    plt.figure(figsize=(8, 8))
    plt.scatter(x_final.cpu().numpy(), y_final.cpu().numpy(),
                s=1, color='blue', alpha=0.5, label='Collocation points')

    theta_arc = np.linspace(0, np.pi / 2, 100)
    plt.plot(R * np.cos(theta_arc), R * np.sin(theta_arc), 'k-', linewidth=2, label='Curved boundary')
    plt.plot([R, x_max], [0, 0], 'k-', linewidth=2)
    plt.plot([x_max, x_max], [0, y_max], 'k-', linewidth=2)
    plt.plot([x_max, 0], [y_max, y_max], 'k-', linewidth=2)
    plt.plot([0, 0], [y_max, R], 'k-', linewidth=2)

    plt.xlabel('x (m)')
    plt.ylabel('y (m)')
    plt.title(f'Point Distribution: Focus on Curve\nFinal points: {len(x_final)}')
    plt.gca().set_aspect('equal')
    plt.legend()
    plt.grid(True, linestyle='--', alpha=0.3)
    plt.show()

    return x_final, y_final

x_pts, y_pts = generate_point_distribution(arc_density=10, arc_influence=1.2)

# PDE

In [ ]:
def PDE(x, y, net):
    # Reshape 1D tensors into column vectors for concatenation
    x_reshaped = x.unsqueeze(1) if x.dim() == 1 else x
    y_reshaped = y.unsqueeze(1) if y.dim() == 1 else y
    
    xy = torch.cat([x_reshaped, y_reshaped], dim=1)
    xy.requires_grad = True
    
    output = net(xy)
    u = output[:, 0:1]
    v = output[:, 1:2]
    
    u_x = torch.autograd.grad(u.sum(), xy, create_graph=True)[0][:, 0].unsqueeze(1)
    u_y = torch.autograd.grad(u.sum(), xy, create_graph=True)[0][:, 1].unsqueeze(1)
    v_x = torch.autograd.grad(v.sum(), xy, create_graph=True)[0][:, 0].unsqueeze(1)
    v_y = torch.autograd.grad(v.sum(), xy, create_graph=True)[0][:, 1].unsqueeze(1)
    
    E = 20  # Young's modulus MPa 
    nu = 0.3 # Poisson's ratio
    gamma = 0 # Distributed load (Mn/m^2)
    lam = E*nu / ((1+nu)*(1-2*nu))   # λ
    mu  = E / (2*(1+nu))              # μ (shear modulus)

    sigma_xx = (lam + 2*mu)*u_x + lam*v_y
    sigma_yy = lam*u_x + (lam + 2*mu)*v_y
    sigma_xy = mu*(u_y + v_x)
    
    f_x = torch.zeros_like(x_reshaped)
    f_y = -gamma * torch.ones_like(y_reshaped)
    
    R_x = torch.autograd.grad(sigma_xx.sum(), xy, create_graph=True)[0][:, 0].unsqueeze(1) + \
          torch.autograd.grad(sigma_xy.sum(), xy, create_graph=True)[0][:, 1].unsqueeze(1) + f_x
    R_y = torch.autograd.grad(sigma_xy.sum(), xy, create_graph=True)[0][:, 0].unsqueeze(1) + \
          torch.autograd.grad(sigma_yy.sum(), xy, create_graph=True)[0][:, 1].unsqueeze(1) + f_y

    loss_x = F.mse_loss(R_x, torch.zeros_like(R_x))
    loss_y = F.mse_loss(R_y, torch.zeros_like(R_y))
    
    return loss_x, loss_y

# Training 


In [ ]:
import torch
import copy
import matplotlib.pyplot as plt
from torch.optim.lr_scheduler import CosineAnnealingLR
from tqdm import tqdm

def train(net, optimizer, n_epochs, fem_data_file=None, nx=100, ny=100):
    scheduler = CosineAnnealingLR(optimizer, T_max=n_epochs, eta_min=1e-4)
    
    best_loss = float('inf')
    
    loss_history = []
    pde_x_history = []
    pde_y_history = []
    bc_history = []
    epoch_history = []
    
    print("Generating training points...")
    x_inner, y_inner = generate_point_distribution(nx=nx, ny=ny)
    print(f"Generated {len(x_inner)} interior points")
    
    print("Generating boundary points...")
    num_boundary_points = 1000 
    
    x_bottom = torch.linspace(1.0, 4.0, num_boundary_points, device=device)
    y_bottom = torch.zeros(num_boundary_points, device=device)
    
    x_left = torch.zeros(num_boundary_points, device=device)
    y_left = torch.linspace(1.0, 4.0, num_boundary_points, device=device)
    
    x_top = torch.linspace(0, 4.0, num_boundary_points, device=device)
    y_top = torch.ones(num_boundary_points, device=device) * 4
    
    x_right = torch.ones(num_boundary_points, device=device) * 4.0
    y_right = torch.linspace(0, 4.0, num_boundary_points, device=device)
    
    theta = torch.linspace(0, torch.pi/2, num_boundary_points, device=device)
    x_curve = torch.cos(theta)
    y_curve = torch.sin(theta)

    x_b = torch.cat([x_bottom, x_left, x_top, x_right, x_curve])
    y_b = torch.cat([y_bottom, y_left, y_top, y_right, y_curve])
    
    xy_b = torch.stack([x_b, y_b], dim=1)
    print(f"Generated {len(xy_b)} boundary points")
    
    print(f"Starting training for {n_epochs} epochs...")
    
    pbar = tqdm(range(n_epochs), desc="Training PINN", unit="epoch")
    
    for epoch in pbar:
        optimizer.zero_grad()
        
        loss_pde_x, loss_pde_y = PDE(x_inner, y_inner, net)
        loss_bc = BC(xy_b, net)
        
        loss = loss_pde_x + loss_pde_y + loss_bc
        
        loss.backward()
        torch.nn.utils.clip_grad_norm_(net.parameters(), max_norm=0.5)
        optimizer.step()
        scheduler.step()
        
        loss_history.append(loss.item())
        pde_x_history.append(loss_pde_x.item())
        pde_y_history.append(loss_pde_y.item())
        bc_history.append(loss_bc.item())
        epoch_history.append(epoch)
        
        if loss < best_loss:
            best_loss = loss
            
            torch.save({
                'net_state_dict': net.state_dict(),
                'optimizer_state_dict': optimizer.state_dict(),
                'scheduler_state_dict': scheduler.state_dict(),
                'loss': loss,
                'epoch': epoch,
            }, 'best_model.pth')
        
        if (epoch + 1) % 10 == 0:
            pbar.set_postfix({
                'Loss': f"{loss.item():.2e}",
                'PDE_x': f"{loss_pde_x.item():.2e}",
                'PDE_y': f"{loss_pde_y.item():.2e}",
                'BC': f"{loss_bc.item():.2e}",
                'LR': f"{scheduler.get_last_lr()[0]:.1e}"
            })
    
    pbar.close()
    
    plt.figure(figsize=(12, 8))
    
    plt.subplot(2, 1, 1)
    plt.plot(epoch_history, loss_history, 'b-', label='Total Loss')
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Loss (log scale)')
    plt.title('Training Loss History')
    plt.legend()
    plt.grid(True)
    
    plt.subplot(2, 1, 2)
    plt.plot(epoch_history, pde_x_history, 'r-', label='PDE_x Loss')
    plt.plot(epoch_history, pde_y_history, 'g-', label='PDE_y Loss')
    plt.plot(epoch_history, bc_history, 'b-', label='BC Loss')
    plt.yscale('log')
    plt.xlabel('Epoch')
    plt.ylabel('Component Losses (log scale)')
    plt.title('Component Losses History')
    plt.legend()
    plt.grid(True)
    
    plt.tight_layout()
    plt.savefig('loss_history.png')
    plt.close()
    
    metrics = {
        'loss_history': loss_history,
        'pde_x_history': pde_x_history,
        'pde_y_history': pde_y_history,
        'bc_history': bc_history,
        'epoch_history': epoch_history
    }
    
    print("Training completed.")
    return net, metrics

# Plot result

In [ ]:
import pandas as pd
import numpy as np
import torch
import matplotlib.pyplot as plt
from matplotlib.patches import Wedge

def plot_results(net, output_filename='Deep_Tunnel_results.csv'):
    nx_plot, ny_plot = 200, 200
    X_lin = torch.linspace(0, 4.0, nx_plot, device=device)
    Y_lin = torch.linspace(0, 4.0, ny_plot, device=device)

    xx, yy = torch.meshgrid(X_lin, Y_lin, indexing='ij')
    X_flat, Y_flat = xx.flatten(), yy.flatten()

    mask    = in_domain(X_flat, Y_flat)
    mask_np = mask.cpu().numpy()

    XY = torch.stack([X_flat, Y_flat], dim=1)
    XY.requires_grad_(True)

    with torch.enable_grad():
        out  = net(XY)
        U, V = out[:, 0:1], out[:, 1:2]
        grads_U = torch.autograd.grad(U.sum(), XY, create_graph=True)[0]
        grads_V = torch.autograd.grad(V.sum(), XY, create_graph=True)[0]
        U_x, U_y = grads_U[:, 0], grads_U[:, 1]
        V_x, V_y = grads_V[:, 0], grads_V[:, 1]
        sigma_xx = C11 * U_x + C12 * V_y
        sigma_yy = C12 * U_x + C11 * V_y
        sigma_xy = C33 * (U_y + V_x)

    X_np      = X_flat.detach().cpu().numpy()
    Y_np      = Y_flat.detach().cpu().numpy()
    U_np      = U.detach().cpu().numpy().squeeze()
    V_np      = V.detach().cpu().numpy().squeeze()
    sxx_np    = sigma_xx.detach().cpu().numpy() * 1000
    syy_np    = sigma_yy.detach().cpu().numpy() * 1000
    sxy_np    = sigma_xy.detach().cpu().numpy() * 1000
    magnitude = np.sqrt(U_np**2 + V_np**2)

    pd.DataFrame({
        'X': X_np[mask_np], 'Y': Y_np[mask_np],
        'ux': U_np[mask_np], 'uy': V_np[mask_np],
        'sigma_xx': sxx_np[mask_np], 'sigma_yy': syy_np[mask_np],
        'sigma_xy': sxy_np[mask_np], 'magnitude': magnitude[mask_np],
    }).to_csv(output_filename, index=False)
    print(f'Results saved to {output_filename}')

    X_1d = X_lin.cpu().numpy()
    Y_1d = Y_lin.cpu().numpy()

    def get_plot_array(arr):
        arr_2d  = arr.reshape(nx_plot, ny_plot)
        mask_2d = mask_np.reshape(nx_plot, ny_plot)
        return np.ma.masked_array(arr_2d.T, mask=~mask_2d.T)

    fields = [
        (U_np,      'Displacement u_x [m]',    'ux'),
        (V_np,      'Displacement u_y [m]',    'uy'),
        (magnitude, 'Displacement |u| [m]',    '|u|'),
        (sxx_np,    'Normal stress σ_xx [kPa]','σxx'),
        (syy_np,    'Normal stress σ_yy [kPa]','σyy'),
        (sxy_np,    'Shear stress σ_xy [kPa]', 'σxy'),
    ]

    fig, axes = plt.subplots(2, 3, figsize=(18, 12))
    for ax, (arr, label, title) in zip(axes.flatten(), fields):
        pcm = ax.pcolormesh(
            X_1d, Y_1d,
            get_plot_array(arr),
            shading='auto', cmap='jet'
        )
        fig.colorbar(pcm, ax=ax, label=label)
        ax.set_title(title)
        ax.set_xlabel('X [m]')
        ax.set_ylabel('Y [m]')
        ax.set_aspect('equal')

        # ── Wedge ปิดบังพื้นที่ภายในอุโมงค์ (สีขาว) ─────────────────────
        tunnel_patch = Wedge(
            center=(0, 0), r=R,
            theta1=0, theta2=90,
            facecolor='white', edgecolor='none', zorder=5
        )
        ax.add_patch(tunnel_patch)

        # ── เส้นขอบ domain ────────────────────────────────────────────────
        theta_arc = np.linspace(0, np.pi/2, 300)
        ax.plot(R*np.cos(theta_arc), R*np.sin(theta_arc), 'k-', lw=1.5, zorder=6)
        ax.plot([R, 4], [0, 0], 'k-', lw=1, zorder=6)
        ax.plot([4, 4], [0, 4], 'k-', lw=1, zorder=6)
        ax.plot([4, 0], [4, 4], 'k-', lw=1, zorder=6)
        ax.plot([0, 0], [4, R], 'k-', lw=1, zorder=6)

        ax.set_xlim(0, 4)
        ax.set_ylim(0, 4)

    plt.suptitle('PINN Results — Deep Tunnel (Plane Strain)', fontsize=14)
    plt.tight_layout()
    plt.savefig('results_plots.png', dpi=300, bbox_inches='tight')
    plt.close()
    print('Result plots saved to results_plots.png')


In [ ]:
# Main execution
if __name__ == "__main__":
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    print(f"Using device: {device}")
    
    # Initialize network and optimizer
    net = Net().to(device)
    optimizer = optim.Adam(net.parameters(), lr=0.001)
    
    # Train the network
    trained_net, metrics = train(net=net, optimizer=optimizer, n_epochs=80000)
    # Plot results
    try:
        plot_results(trained_net)
    except Exception as e:
        print(f"Error in plot_results: {str(e)}")